In [ ]:
import os

import duckdb

# parameters
scale_factor = 10  # TPC-H scale factor (1 = ~1GB)
output_dir = f"/mnt/labstore/bespoke_olap/tpch_parquet/sf{scale_factor}"
os.makedirs(output_dir, exist_ok=True)

# connect and load tpch generator
con = duckdb.connect()
con.execute("INSTALL tpch; LOAD tpch;")

# list of TPC-H tables
tables = [
    "region",
    "nation",
    "supplier",
    "customer",
    "part",
    "partsupp",
    "orders",
    "lineitem",
]

# generate each table and export as Parquet
print(f"Generating TPC-H data at scale factor {scale_factor} ...")
con.execute(f"CALL dbgen(sf={scale_factor});")  # generates all tables in memory
for t in tables:
    print(f"Saving {t} ...")
    con.execute(
        f"COPY (SELECT * FROM {t}) TO '{output_dir}/{t}.parquet' (FORMAT PARQUET);"
    )

print(f"\n✅ Parquet files stored in: {output_dir}/")

# Also save as .duckdb file
db_file = f"{output_dir}/duckdb.db"
db_con = duckdb.connect(db_file)
for t in tables:
    print(f"Copying {t} to DuckDB file ...")
    db_con.execute(
        f"CREATE TABLE {t} AS SELECT * FROM parquet_scan('{output_dir}/{t}.parquet');"
    )
db_con.close()

print(f"\n✅ DuckDB file stored at: {db_file}")